# AS-IS · Ciencia de Datos: recompra en cualquier momento

Este notebook concentra **toda la lógica analítica** del caso: exploración, construcción de etiquetas, ingeniería de variables, preparación, entrenamiento, evaluación y comparación de modelos. **No utiliza** módulos de `src/`, DVC, Feast, MLflow, Airflow, ZenML ni servicios Docker.

**Problema:** dado un cliente que ya ha realizado al menos una compra **hasta una fecha de corte mensual** $t$, predecir si realizará **otra compra en los próximos 60 días**: $Y_{i,t} = 1\{\exists\; \mathrm{compra}_i \in (t,t+60\,d]\}$. El cliente puede aparecer en varios snapshots, sin vincular la predicción únicamente a los 60 días posteriores a su primera compra.

**Datos:** se generan CSV de clientes y transacciones de forma sintética con `scripts/generate_transactions.py`. El simulador utiliza perfiles latentes separados que no deben entrar al modelo.

## 1. Entorno y parámetros

El notebook puede abrirse desde la raíz o desde `notebooks/`. Se apoyará solo en el generador externo y bibliotecas de ciencia de datos instaladas con `uv sync`.

In [ ]:
from pathlib import Path
import sys
import subprocess
from bisect import bisect_right
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "scripts" / "generate_transactions.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Abre el notebook dentro del proyecto generado.")

DATA_ROOT = ROOT / "data" / "synthetic"
SEED = 42
HORIZON_DAYS = 60
TEST_FRACTION = 0.25
LABEL = "repurchase_next_60d"

print("Raíz del proyecto:", ROOT)

## 2. Generar los datos observables

La primera ejecución genera **12 meses**, 1,000 clientes iniciales y nuevas altas durante los meses siguientes. Si los CSV ya existen, se reutilizan. Puedes controlar el histórico usando los parámetros del generador desde la terminal.

**No leer `simulation/customer_profiles.csv` para construir el modelo**: representa variables latentes que el negocio no observaría.

In [ ]:
customers_file = DATA_ROOT / "raw" / "customers.csv"
transaction_files = sorted((DATA_ROOT / "raw" / "transactions").glob("year=*/month=*/transactions.csv"))

if not customers_file.exists():
    if transaction_files:
        raise RuntimeError("Existen transacciones, pero falta customers.csv: revisa el estado del generador.")
    subprocess.run(
        [
            sys.executable, str(ROOT / "scripts" / "generate_transactions.py"),
            "--init", "--start", "2025-01", "--months", "12",
            "--customers", "1000", "--seed", str(SEED),
        ],
        cwd=ROOT,
        check=True,
    )
    transaction_files = sorted((DATA_ROOT / "raw" / "transactions").glob("year=*/month=*/transactions.csv"))

if not transaction_files:
    raise FileNotFoundError("No existen transacciones en data/synthetic/raw/transactions/.")
print(f"Particiones mensuales: {len(transaction_files)}")

## 3. Ingesta y exploración (EDA)

Los insumos del científico de datos son `customers.csv` y los CSV mensuales de `transactions/`. Exploramos esquema, completitud, fechas, monedas, categorías, canales y actividad.

In [ ]:
customers = pd.read_csv(customers_file, dtype={"customer_id": "string"})
transactions = pd.concat(
    [pd.read_csv(path, dtype={"customer_id": "string"}) for path in transaction_files],
    ignore_index=True,
)

customers["created_at"] = pd.to_datetime(customers["created_at"])
customers["birth_date"] = pd.to_datetime(customers["birth_date"])
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"])

print("Clientes:", customers.shape)
print("Transacciones:", transactions.shape)
display(customers.head())
display(transactions.head())

In [ ]:
print("CLIENTES")
customers.info()
display(customers.describe(include="all"))
print("TRANSACCIONES")
transactions.info()
display(transactions.describe(include="all"))
print("Nulos: clientes")
display(customers.isna().sum().to_frame("nulls"))
print("Nulos: transacciones")
display(transactions.isna().sum().to_frame("nulls"))

In [ ]:
assert customers["customer_id"].is_unique, "Clientes duplicados"
assert transactions["transaction_id"].is_unique, "Transacciones duplicadas"
assert transactions["customer_id"].isin(customers["customer_id"]).all(), "Transacciones huérfanas"
assert (transactions["amount"] >= 0).all(), "Importes negativos"
assert (transactions["items_count"] > 0).all(), "Cantidad de artículos inválida"
assert not transactions.isna().any().any(), "Transacciones con nulos"

print("Fecha alta clientes:", customers["created_at"].min().date(), "a", customers["created_at"].max().date())
print("Fecha transacciones:", transactions["transaction_date"].min().date(), "a", transactions["transaction_date"].max().date())
print("Monedas:", transactions["currency"].value_counts().to_dict())
print("Categorías:", transactions["category"].value_counts().to_dict())

In [ ]:
monthly = (
    transactions.assign(month=transactions["transaction_date"].dt.to_period("M").astype(str))
    .groupby("month", as_index=False)
    .agg(transactions=("transaction_id", "count"), revenue=("amount", "sum"),
         active_customers=("customer_id", "nunique"))
    .sort_values("month")
)
display(monthly)
fig, axs = plt.subplots(1, 2, figsize=(13, 4))
axs[0].plot(monthly["month"], monthly["transactions"], marker="o")
axs[0].set_title("Transacciones mensuales")
axs[1].plot(monthly["month"], monthly["revenue"], marker="o")
axs[1].set_title("Ingresos mensuales (PEN)")
for ax in axs:
    ax.tick_params(axis="x", rotation=45)
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4))
transactions["amount"].hist(ax=axs[0], bins=35)
axs[0].set_title("Distribución de importe")
transactions["sales_channel"].value_counts().plot.bar(ax=axs[1])
axs[1].set_title("Canal de venta")
transactions["payment_method"].value_counts().plot.bar(ax=axs[2])
axs[2].set_title("Medio de pago")
plt.tight_layout()
plt.show()

In [ ]:
frequency = transactions.groupby("customer_id").size()
print("Clientes con al menos una compra:", frequency.size)
print("Clientes sin compras:", customers.customer_id.nunique() - frequency.size)
display(frequency.describe().to_frame("compras_por_cliente"))
ax = frequency.hist(bins=35, figsize=(8, 4))
ax.set_title("Frecuencia observada de compras por cliente")
ax.set_xlabel("Número de transacciones")
plt.show()

## 4. Formulación temporal y snapshots

Creamos un snapshot por cliente y cierre de mes. **Las features usan solo eventos anteriores o iguales a $t$**; el target utiliza compras posteriores al corte, hasta $t+60$ días.

Para entrenar, se incluyen únicamente snapshots de clientes con compras observadas **antes o en el corte** y cuya ventana futura esté completamente cubierta por los meses generados (evitamos censura a la derecha).

La fecha `event_timestamp` será la medianoche del día siguiente al cierre, útil para una integración point-in-time posterior, pero aquí no necesitamos Feast.

In [ ]:
# Organización de transacciones en memoria para consultar ventanas eficientemente.
transactions_sorted = transactions.sort_values(["customer_id", "transaction_date", "transaction_id"])
tx_by_customer = {
    str(customer_id): group.to_dict("records")
    for customer_id, group in transactions_sorted.groupby("customer_id")
}
dates_by_customer = {
    customer_id: [row["transaction_date"] for row in rows]
    for customer_id, rows in tx_by_customer.items()
}

periods = pd.period_range(
    transactions["transaction_date"].min().to_period("M"),
    transactions["transaction_date"].max().to_period("M"),
    freq="M",
)
coverage_end = periods[-1].end_time.normalize()
print("Meses observados:", len(periods), "| Cobertura hasta:", coverage_end.date())

In [ ]:
def preferred_value(values):
    """Modo determinista; NaN si no hay observaciones."""
    if not values:
        return np.nan
    counts = Counter(values)
    return sorted(counts, key=lambda v: (-counts[v], str(v)))[0]


def age_at(birth_date, cutoff):
    return cutoff.year - birth_date.year - (
        (cutoff.month, cutoff.day) < (birth_date.month, birth_date.day)
    )


def build_snapshot(customer, cutoff, coverage_end, horizon_days=60):
    """Una fila por cliente/fecha: historia <= corte, futuro solo para el target."""
    customer_id = str(customer["customer_id"])
    if customer["created_at"] > cutoff:
        return None
    events = tx_by_customer.get(customer_id, [])
    dates = dates_by_customer.get(customer_id, [])
    hi = bisect_right(dates, cutoff)

    def window(start_exclusive, end_inclusive):
        lo = bisect_right(dates, start_exclusive)
        end = bisect_right(dates, end_inclusive)
        return events[lo:end]

    recent30 = window(cutoff - pd.Timedelta(days=30), cutoff)
    recent90 = window(cutoff - pd.Timedelta(days=90), cutoff)
    previous30 = window(cutoff - pd.Timedelta(days=60), cutoff - pd.Timedelta(days=30))
    spend30 = sum(float(x["amount"]) for x in recent30)
    spend90 = sum(float(x["amount"]) for x in recent90)
    spend_prev30 = sum(float(x["amount"]) for x in previous30)
    n90 = len(recent90)
    last_purchase = dates[hi-1] if hi else None

    # La etiqueta se construye únicamente cuando la ventana está madura.
    # Sin compra previa al corte no modelamos una recompra.
    is_mature = cutoff + pd.Timedelta(days=horizon_days) <= coverage_end
    target = np.nan
    if hi > 0 and is_mature:
        idx_future = bisect_right(dates, cutoff + pd.Timedelta(days=horizon_days))
        target = int(idx_future > hi)

    return {
        "customer_id": customer_id,
        "snapshot_date": cutoff,
        "event_timestamp": (cutoff + pd.Timedelta(days=1)).tz_localize("UTC"),
        "age": age_at(customer["birth_date"], cutoff),
        "customer_tenure_days": (cutoff - customer["created_at"]).days,
        "has_purchase_history": int(hi > 0),
        "days_since_last_purchase": (cutoff - last_purchase).days if last_purchase is not None else np.nan,
        "purchases_30d": len(recent30),
        "purchases_90d": n90,
        "spend_30d": round(spend30, 2),
        "spend_90d": round(spend90, 2),
        "avg_ticket_90d": round(spend90/n90, 2) if n90 else np.nan,
        "items_90d": sum(int(x["items_count"]) for x in recent90),
        "avg_items_per_purchase_90d": round(sum(int(x["items_count"]) for x in recent90)/n90, 3) if n90 else np.nan,
        "category_diversity_90d": len({x["category"] for x in recent90}),
        "preferred_category_90d": preferred_value([x["category"] for x in recent90]),
        "preferred_payment_method_90d": preferred_value([x["payment_method"] for x in recent90]),
        "digital_payment_ratio_90d": round(sum(x["payment_method"] == "digital_wallet" for x in recent90)/n90, 4) if n90 else np.nan,
        "online_purchase_ratio_90d": round(sum(x["sales_channel"] in ("app", "web") for x in recent90)/n90, 4) if n90 else np.nan,
        "discount_usage_ratio_90d": round(sum(float(x["discount_amount"]) > 0 for x in recent90)/n90, 4) if n90 else np.nan,
        "discount_total_90d": round(sum(float(x["discount_amount"]) for x in recent90), 2),
        "spend_previous_30d": round(spend_prev30, 2),
        "spend_delta_30d": round(spend30 - spend_prev30, 2),
        LABEL: target,
    }

In [ ]:
rows = []
for period in periods:
    cutoff = period.end_time.normalize()
    for customer in customers.to_dict("records"):
        row = build_snapshot(customer, cutoff, coverage_end, HORIZON_DAYS)
        if row is not None:
            rows.append(row)

snapshots = pd.DataFrame(rows)
assert not snapshots.duplicated(["customer_id", "snapshot_date"]).any()
print("Total snapshots observables:", len(snapshots))
print("Snapshots con target maduro:", snapshots[LABEL].notna().sum())
display(snapshots.head())

## 5. Construcción y auditoría del target

Un cliente puede participar en múltiples snapshots con una etiqueta distinta en cada uno. No usamos información futura como predictor; los únicos campos derivados del futuro se limitan a la etiqueta durante la preparación supervisada.

In [ ]:
model_df = snapshots.loc[
    snapshots[LABEL].notna() & snapshots["has_purchase_history"].eq(1)
].copy()
model_df[LABEL] = model_df[LABEL].astype(int)

summary = (
    model_df.groupby("snapshot_date", as_index=False)
    .agg(observations=("customer_id", "size"), positive_rate=(LABEL, "mean"))
)
print("Clientes únicos:", model_df.customer_id.nunique())
print("Observaciones supervisadas:", len(model_df))
print("Tasa global de recompra:", round(float(model_df[LABEL].mean()), 4))
display(summary)

In [ ]:
# Comprobación conceptual: un mismo cliente puede aparecer en varios cortes.
repeats = model_df.groupby("customer_id")["snapshot_date"].nunique()
print("Clientes con múltiples snapshots:", int((repeats > 1).sum()))
changes = model_df.groupby("customer_id")[LABEL].nunique()
print("Clientes con cambio de etiqueta entre cortes:", int((changes > 1).sum()))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(summary["snapshot_date"], summary["positive_rate"], marker="o")
ax.set_title("Tasa de recompra futura por corte mensual")
ax.set_ylabel("Proporción positiva")
ax.set_ylim(0, 1)
ax.grid(alpha=0.25)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Split temporal con embargo de 60 días

**No aplicamos `train_test_split` aleatorio:** las filas de un mismo cliente a lo largo del tiempo están relacionadas y el target utiliza una ventana futura. Separamos por fecha de snapshot, reservando el 25% final de cortes para test y eliminando de train las observaciones cuyo horizonte de etiqueta invadiría test.

In [ ]:
unique_dates = sorted(model_df["event_timestamp"].unique())
cut_index = max(1, min(len(unique_dates)-1, int(len(unique_dates)*(1 - TEST_FRACTION))))
test_start = pd.Timestamp(unique_dates[cut_index])
train_cutoff = test_start - pd.Timedelta(days=HORIZON_DAYS)

train_df = model_df.loc[model_df["event_timestamp"] <= train_cutoff].copy()
test_df = model_df.loc[model_df["event_timestamp"] >= test_start].copy()
assert len(train_df) > 0 and len(test_df) > 0
assert set(train_df[LABEL].unique()) == {0, 1}
assert set(test_df[LABEL].unique()) == {0, 1}
assert train_df.event_timestamp.max() + pd.Timedelta(days=HORIZON_DAYS) <= test_start

print("Training:", train_df.shape, "| último snapshot:", train_df.snapshot_date.max().date())
print("Evaluation:", test_df.shape, "| primer snapshot:", test_df.snapshot_date.min().date())
print("Embargo:", HORIZON_DAYS, "días")
print("Tasas positivas:", round(train_df[LABEL].mean(),3), round(test_df[LABEL].mean(),3))

## 7. Preprocesamiento sin fuga de datos

Excluimos `customer_id`, timestamps y el target. El `has_purchase_history` también se excluye porque, tras filtrar los compradores existentes, es constante. **Imputación, codificación y escalamiento se ajustan solo con training** dentro de cada `Pipeline` de scikit-learn.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NON_PREDICTIVE = {"customer_id", "snapshot_date", "event_timestamp", LABEL, "has_purchase_history"}
feature_columns = [c for c in model_df.columns if c not in NON_PREDICTIVE]
categorical_features = ["preferred_category_90d", "preferred_payment_method_90d"]
numerical_features = [c for c in feature_columns if c not in categorical_features]

X_train, y_train = train_df[feature_columns], train_df[LABEL]
X_test, y_test = test_df[feature_columns], test_df[LABEL]

print("Features:", len(feature_columns))
print("Numéricas:", numerical_features)
print("Categóricas:", categorical_features)

In [ ]:
def make_preprocessor(scale_numeric=False):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    numeric_pipeline = Pipeline(numeric_steps)
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("num", numeric_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features),
    ])

## 8. Entrenamiento de modelos baseline

Igual que en el notebook AS-IS original, comparamos **Logistic Regression**, **Random Forest** y **XGBoost**. Solo Logistic Regression emplea `StandardScaler` para las columnas numéricas; los modelos basados en árboles no lo requieren.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay,
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=150, max_depth=10, random_state=SEED, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.1,
                             subsample=0.9, colsample_bytree=0.9,
                             random_state=SEED, n_jobs=2, eval_metric="logloss", tree_method="hist"),
}
trained = {}
metrics = []

In [ ]:
for model_name, estimator in models.items():
    pipeline = Pipeline([
        ("preprocessor", make_preprocessor(scale_numeric=model_name == "Logistic Regression")),
        ("model", estimator),
    ])
    pipeline.fit(X_train, y_train)
    prob = pipeline.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)
    trained[model_name] = (pipeline, prob, pred)
    metrics.append({
        "model": model_name,
        "roc_auc": roc_auc_score(y_test, prob),
        "average_precision": average_precision_score(y_test, prob),
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
    })
    print(f"Entrenado: {model_name}")

## 9. Evaluación y comparación

Observamos métricas de clasificación, ROC, Precision-Recall y matrices de confusión sobre **el mismo conjunto de test futuro**. La selección por ROC AUC o Average Precision depende de la prioridad de negocio; no interpretamos estas métricas como resultados de producción.

In [ ]:
comparison = pd.DataFrame(metrics).set_index("model").sort_values("average_precision", ascending=False)
display(comparison.round(4))
print("Mejor baseline según Average Precision:", comparison.index[0])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for model_name, (_, probabilities, _) in trained.items():
    RocCurveDisplay.from_predictions(y_test, probabilities, name=model_name, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_test, probabilities, name=model_name, ax=axes[1])
axes[0].set_title("Curvas ROC")
axes[1].set_title("Curvas Precision-Recall")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (model_name, (_, _, preds)) in zip(axes, trained.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, preds, ax=ax, colorbar=False)
    ax.set_title(model_name)
plt.tight_layout()
plt.show()

## 10. Interpretación de variables

Como aproximación exploratoria, inspeccionamos la importancia integrada de XGBoost para las columnas resultantes del `ColumnTransformer`. Esta importancia **no demuestra causalidad**.

In [ ]:
xgb_pipe = trained["XGBoost"][0]
feature_names = xgb_pipe.named_steps["preprocessor"].get_feature_names_out()
importances = xgb_pipe.named_steps["model"].feature_importances_
importance_df = pd.DataFrame({"feature": feature_names, "importance": importances})
importance_df = importance_df.sort_values("importance", ascending=False)
display(importance_df.head(15))
plot_data = importance_df.head(15).sort_values("importance")
ax = plot_data.plot.barh(x="feature", y="importance", legend=False, figsize=(9, 6))
ax.set_title("Importancia de variables · XGBoost")
plt.tight_layout()
plt.show()

## 11. Exportación local de artefactos (opcional)

Guardamos modelos entrenados como `joblib` **desde el propio notebook**, para que puedan migrarse posteriormente a pipelines dedicados. Los archivos generados se excluyen de Git; no se integra aquí ningún registro ni servidor de artefactos.

In [ ]:
import joblib

artifact_dir = ROOT / "artifacts"
artifact_dir.mkdir(exist_ok=True)
filenames = {
    "Logistic Regression": "logistic_regression.joblib",
    "Random Forest": "random_forest.joblib",
    "XGBoost": "xgboost.joblib",
}
for name, (model, _, _) in trained.items():
    destination = artifact_dir / filenames[name]
    joblib.dump(model, destination)
    print("Modelo guardado:", destination.relative_to(ROOT))

## 12. Conclusiones y transición hacia MLOps

- La unidad de observación es **(cliente, cierre de mes)**, no «cliente después de su primera compra».
- La etiqueta solo se conoce cuando han transcurrido sus 60 días completos; snapshots recientes se excluyen del entrenamiento.
- Las features son calculadas **exclusivamente con compras anteriores o iguales al corte**.
- Se utiliza un **split temporal con embargo de 60 días**, no una partición aleatoria.
- La imputación, la codificación y el escalamiento se ajustan con training mediante `sklearn.Pipeline`.
- Este notebook es intencionalmente **AS-IS**: la clase irá extrayendo funciones hacia `src/`, configuraciones y entrypoints; los archivos de infraestructura están vacíos.

**Extensión didáctica:** usa `--advance` en el generador, vuelve a ejecutar las celdas y compara la estabilidad de los baselines al incorporar datos más recientes. Cada ejecución debe partir de los CSV observables, nunca de `simulation/customer_profiles.csv`.